# SAJU ML NEW_DEV_2 Frozen TENGOD Replication V3

**Notebook version:** `SAJU_ML_NEW_DEV_2_FROZEN_TENGOD_REPLICATION_V3_20260815`

## Purpose

One-shot external replication of the architecture frozen by the V3 all-feature tournament.

The only allowed architecture is:

- `TENGOD_FIXED`
- 10 frozen Daewoon/Sewoon grouped Ten-God features
- Pairwise Logistic Regression
- L2
- `C = 0.3`

## Fixed discovery refit

The architecture is reconstructed on the already-observed discovery pool:

- V1 Career pair-evaluable: 23 subjects
- V2 NEW_DEV: 70 subjects
- total: 93 subjects

This refit only reconstructs coefficients/scaler for the **already frozen architecture**.
It is not model selection or tuning.

## Untouched replication set

- NEW_DEV_2: 100 new subjects
- exactly 1 Career-positive + 1 Career-negative year per person
- exactly 100 independent ranking pairs

## Forbidden

- no feature search
- no hyperparameter search
- no model search
- no coefficient-sign hand tuning
- no NEW_CONFIRM loading
- no Public CHECK / FINAL loading
- no Validation B loading

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_NEW_DEV_2_FROZEN_TENGOD_REPLICATION_V3_20260815"
SEED = 20260815
N_PERMUTATIONS = 5000
N_BOOTSTRAP = 20000

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing saju_engine.py. "
        "Run this notebook from inside the Chartpalja saju repo."
    )

REPO_ROOT = find_repo_root()

NEW_DIR = REPO_ROOT / "research/ml_corpus/new_dev_2"
NEW_CORPUS_PATH = NEW_DIR / "SAJU_ML_NEW_DEV_2_CORPUS.json"
NEW_ENGINE_PATH = NEW_DIR / "SAJU_ML_NEW_DEV_2_ENGINE_INPUTS.json"
NEW_MANIFEST_PATH = NEW_DIR / "SAJU_ML_NEW_DEV_2_MANIFEST.json"

FREEZE_PATH = (
    REPO_ROOT
    / "research/ml/artifacts/v3_all_feature_tournament/"
      "PROVISIONAL_ARCHITECTURE_FREEZE_V3.json"
)

V1_CORPUS_PATH = REPO_ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json"
V1_FEATURE_DIR = REPO_ROOT / "research/ml/artifacts/v1"

V2_DEV_CORPUS_PATH = (
    REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json"
)
V2_FEATURE_PATH = (
    REPO_ROOT / "research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv"
)

OUT = REPO_ROOT / "research/ml/artifacts/v3_new_dev_2_replication"
OUT.mkdir(parents=True, exist_ok=True)

FEATURE_CACHE = OUT / "NEW_DEV_2_feature_table_v3.csv"

print("=" * 80)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("output:", OUT)
print("=" * 80)

NOTEBOOK VERSION: SAJU_ML_NEW_DEV_2_FROZEN_TENGOD_REPLICATION_V3_20260815
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v3_new_dev_2_replication


## 1. Frozen preflight and holdout guard

In [2]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

required = [
    NEW_CORPUS_PATH,
    NEW_ENGINE_PATH,
    NEW_MANIFEST_PATH,
    FREEZE_PATH,
    V1_CORPUS_PATH,
    V2_DEV_CORPUS_PATH,
    V2_FEATURE_PATH,
]

for path in required:
    ok = path.exists()
    print("OK" if ok else "MISSING", path)
    if not ok:
        raise FileNotFoundError(path)

v1_pq = V1_FEATURE_DIR / "ml_feature_table_v1.parquet"
v1_csv = V1_FEATURE_DIR / "ml_feature_table_v1.csv"

if not v1_pq.exists() and not v1_csv.exists():
    raise FileNotFoundError(
        "Need research/ml/artifacts/v1/ml_feature_table_v1.parquet or .csv"
    )

manifest = json.load(open(NEW_MANIFEST_PATH, encoding="utf-8"))
freeze = json.load(open(FREEZE_PATH, encoding="utf-8"))
new_corpus = json.load(open(NEW_CORPUS_PATH, encoding="utf-8"))
new_engine = json.load(open(NEW_ENGINE_PATH, encoding="utf-8"))

assert (
    sha256_file(NEW_CORPUS_PATH)
    == manifest["files"][NEW_CORPUS_PATH.name]["sha256"]
)
assert (
    sha256_file(NEW_ENGINE_PATH)
    == manifest["files"][NEW_ENGINE_PATH.name]["sha256"]
)

assert freeze["status"] == "DISCOVERY_ARCHITECTURE_FROZEN_FOR_NEW_DEV_2"
assert freeze["winner"] == "TENGOD_FIXED"
assert freeze["winner_parameters"]["penalty"] == "l2"
assert float(freeze["winner_parameters"]["C"]) == 0.3
assert int(freeze["n_selected_features"]) == 10
assert all(x.startswith("tengod__") for x in freeze["selected_features"])
assert freeze["discovery_pool"]["NEW_CONFIRM_loaded"] is False

assert new_corpus["status"].startswith("FROZEN_UNTOUCHED_EXTERNAL_REPLICATION_CORPUS")
assert len(new_corpus["subjects"]) == 100
assert len(new_engine["subjects"]) == 100

# Every replication subject must have two distinct Career event years.
same_year_subjects = []
for subject in new_corpus["subjects"]:
    career_events = [
        e for e in subject.get("events", [])
        if (e.get("targets") or {}).get("Career") in ("positive", "negative")
        and e.get("ml_usable", True)
    ]
    years = [int(e["year"]) for e in career_events]
    if len(years) != 2 or len(set(years)) != 2:
        same_year_subjects.append((subject["subject_id"], subject["name"], years))

if same_year_subjects:
    raise RuntimeError(
        "NEW_DEV_2 contains non-distinct Career event years: %s"
        % same_year_subjects
    )

TENGOD_COLS = list(freeze["selected_features"])

print("\nFROZEN ARCHITECTURE PREFLIGHT PASSED")
print("winner:", freeze["winner"])
print("n_features:", len(TENGOD_COLS))
print("model:", freeze["winner_parameters"])
print("NEW_DEV_2 subjects:", len(new_corpus["subjects"]))
print("NEW_CONFIRM loaded: False")

OK /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_ENGINE_INPUTS.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_MANIFEST.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v3_all_feature_tournament/PROVISIONAL_ARCHITECTURE_FREEZE_V3.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json
OK /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv

FROZEN ARCHITECTURE PREFLIGHT PASSED
winner: TENGOD_FIXED
n_features: 10
model: {'model': 'Pairwise Logistic Regression', 'penalty': 'l2', 'C': 0.3}
NEW_DEV_2 subjects: 100
NEW_CONFIRM loaded: False


## 2. Build Career subject-year labels

Rules are unchanged:

- unknown is excluded
- conflicting labels in the same subject-year are removed
- only subjects with both a positive and negative Career year are pair-evaluable

For NEW_DEV_2 the corpus additionally requires exactly 2 rows per subject:
one positive and one negative.

In [3]:
def build_target_rows(corpus, pool_label):
    raw = []

    for subject in corpus["subjects"]:
        if not subject.get("birth", {}).get("birth_time_known", True):
            continue

        for event in subject.get("events", []):
            if not event.get("ml_usable", True):
                continue

            label = (event.get("targets") or {}).get("Career", "unknown")

            if label not in ("positive", "negative"):
                continue

            raw.append({
                "subject_id": subject["subject_id"],
                "name": subject["name"],
                "gender": subject.get("gender", "unknown"),
                "occupation_category": subject.get(
                    "occupation_category", "unknown"
                ),
                "pool": pool_label,
                "year": int(event["year"]),
                "label": 1 if label == "positive" else 0,
                "confidence": event.get(
                    "source_event_confidence", "unknown"
                ),
            })

    raw_df = pd.DataFrame(raw)
    collapsed = []

    for (sid, year), group in raw_df.groupby(
        ["subject_id", "year"]
    ):
        if group["label"].nunique() > 1:
            continue

        collapsed.append({
            "subject_id": sid,
            "name": group["name"].iloc[0],
            "gender": group["gender"].iloc[0],
            "occupation_category": group[
                "occupation_category"
            ].iloc[0],
            "pool": pool_label,
            "year": int(year),
            "label": int(group["label"].iloc[0]),
            "confidence": group["confidence"].iloc[0],
        })

    out = pd.DataFrame(collapsed)

    pairable = []

    for sid, group in out.groupby("subject_id"):
        if set(group["label"].tolist()) == set([0, 1]):
            pairable.append(sid)

    return out[out["subject_id"].isin(pairable)].copy()

v1_corpus = json.load(open(V1_CORPUS_PATH, encoding="utf-8"))
v2_corpus = json.load(open(V2_DEV_CORPUS_PATH, encoding="utf-8"))

v1_targets = build_target_rows(v1_corpus, "V1")
v2_targets = build_target_rows(v2_corpus, "V2_NEW_DEV")
new_targets = build_target_rows(new_corpus, "NEW_DEV_2")

print("V1 pairable:", v1_targets["subject_id"].nunique())
print("V2 NEW_DEV pairable:", v2_targets["subject_id"].nunique())
print("NEW_DEV_2 pairable:", new_targets["subject_id"].nunique())

assert v1_targets["subject_id"].nunique() == 23
assert v2_targets["subject_id"].nunique() == 70
assert new_targets["subject_id"].nunique() == 100

counts = (
    new_targets
    .groupby("subject_id")["label"]
    .agg(["count", "sum"])
)

assert (counts["count"] == 2).all()
assert (counts["sum"] == 1).all()

print("NEW_DEV_2 exact 1-positive / 1-negative check: PASS")

V1 pairable: 23
V2 NEW_DEV pairable: 70
NEW_DEV_2 pairable: 100
NEW_DEV_2 exact 1-positive / 1-negative check: PASS


## 3. Refit the exact frozen architecture on the fixed 93-subject discovery pool

In [4]:
if v1_pq.exists():
    v1_features = pd.read_parquet(v1_pq)
else:
    v1_features = pd.read_csv(v1_csv)

v2_features = pd.read_csv(V2_FEATURE_PATH)

for frame_name, frame in [
    ("V1", v1_features),
    ("V2_NEW_DEV", v2_features),
]:
    missing = [
        c for c in TENGOD_COLS
        if c not in frame.columns
    ]

    if missing:
        raise RuntimeError(
            "%s missing frozen TENGOD features: %s"
            % (frame_name, missing)
        )

v1_train = v1_targets.merge(
    v1_features[
        ["subject_id", "year"] + TENGOD_COLS
    ],
    on=["subject_id", "year"],
    how="inner",
    validate="one_to_one",
)

v2_train = v2_targets.merge(
    v2_features[
        ["subject_id", "year"] + TENGOD_COLS
    ],
    on=["subject_id", "year"],
    how="inner",
    validate="one_to_one",
)

discovery = pd.concat(
    [v1_train, v2_train],
    ignore_index=True,
)

assert discovery["subject_id"].nunique() == 93

def make_pairwise_training(frame, columns):
    X_rows = []
    y_rows = []

    for sid, group in frame.groupby("subject_id"):
        positive = group[group["label"] == 1]
        negative = group[group["label"] == 0]

        for _, p_row in positive.iterrows():
            for _, n_row in negative.iterrows():
                diff = (
                    p_row[columns].to_numpy(dtype=float)
                    - n_row[columns].to_numpy(dtype=float)
                )

                X_rows.append(diff)
                y_rows.append(1)

                X_rows.append(-diff)
                y_rows.append(0)

    return np.asarray(X_rows), np.asarray(y_rows)

MODEL = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        C=0.3,
        max_iter=5000,
        random_state=SEED,
    )),
])

X_train, y_train = make_pairwise_training(
    discovery,
    TENGOD_COLS,
)

MODEL.fit(X_train, y_train)

print("Frozen discovery refit complete")
print("subjects:", discovery["subject_id"].nunique())
print("pairwise training rows:", len(y_train))

Frozen discovery refit complete
subjects: 93
pairwise training rows: 588


## 4. NEW_DEV_2 engine adapter

Birth records contain civil date/time, UTC offset and longitude.

To preserve the V2/V3 engine convention:

1. calculate longitude-based solar-time correction using `sajupy`
2. adjust civil date/time once
3. pass the corrected values to `BirthInput`
4. set `use_solar_time=False` to avoid double correction

In [5]:
sys.path.insert(0, str(REPO_ROOT))

import saju_engine as se
import sajupy

STEMS = list("甲乙丙丁戊己庚辛壬癸")
BRANCHES = list("子丑寅卯辰巳午未申酉戌亥")

STEM_ELEMENT_EN = {
    "甲":"wood", "乙":"wood",
    "丙":"fire", "丁":"fire",
    "戊":"earth", "己":"earth",
    "庚":"metal", "辛":"metal",
    "壬":"water", "癸":"water",
}

GENERATES = {
    "wood":"fire",
    "fire":"earth",
    "earth":"metal",
    "metal":"water",
    "water":"wood",
}

CONTROLS = {
    "wood":"earth",
    "earth":"water",
    "water":"fire",
    "fire":"metal",
    "metal":"wood",
}

def parse_utc_offset(raw):
    if isinstance(raw, (int, float)):
        return float(raw)

    s = str(raw).strip()
    sign = -1.0 if s.startswith("-") else 1.0
    s = s[1:] if s[:1] in "+-" else s

    hh, mm = s.split(":")
    return sign * (
        int(hh) + int(mm) / 60.0
    )

def corrected_birth(public_birth):
    y, m, d = map(
        int,
        public_birth["date"].split("-"),
    )
    hh, mi = map(
        int,
        public_birth["time"].split(":")[:2],
    )

    lon = float(public_birth["longitude"])
    utc_hours = parse_utc_offset(
        public_birth["utc_offset"]
    )

    calc = sajupy.get_saju_calculator()
    civil = datetime(y, m, d, hh, mi)

    correction = float(
        calc._calculate_solar_time_correction(
            lon,
            utc_hours,
        )
    )

    h2, min2, day_change = (
        calc._adjust_time_for_solar(
            civil.hour,
            civil.minute,
            correction,
        )
    )

    y2, m2, d2 = (
        calc._adjust_date_for_solar(
            civil.year,
            civil.month,
            civil.day,
            day_change,
        )
    )

    return {
        "y": y2,
        "m": m2,
        "d": d2,
        "h": h2,
        "min": min2,
    }

def pillar_chars(value):
    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
    ):
        a, b = str(value[0]), str(value[1])

        if a in STEMS and b in BRANCHES:
            return a, b

    s = str(value)

    stem = next(
        (c for c in s if c in STEMS),
        None,
    )
    branch = next(
        (c for c in s if c in BRANCHES),
        None,
    )

    return stem, branch

def sexagenary_year_pillar(year):
    i = (int(year) - 1984) % 60
    return (
        STEMS[i % 10]
        + BRANCHES[i % 12]
    )

def ten_god_group(day_stem, other_stem):
    day_element = STEM_ELEMENT_EN[day_stem]
    other_element = STEM_ELEMENT_EN[other_stem]

    if day_element == other_element:
        return "peer"

    if GENERATES[day_element] == other_element:
        return "output"

    if CONTROLS[day_element] == other_element:
        return "wealth"

    if CONTROLS[other_element] == day_element:
        return "officer"

    if GENERATES[other_element] == day_element:
        return "resource"

    raise RuntimeError(
        (day_stem, other_stem)
    )

def get_daewoon_pillar(meta_row):
    for key in (
        "대운_pillar",
        "대운",
        "daewoon_pillar",
    ):
        if meta_row.get(key):
            stem, branch = pillar_chars(
                meta_row[key]
            )

            if stem and branch:
                return stem + branch

    return None

def get_sewoon_pillar(meta_row, year):
    for key in (
        "세운_pillar",
        "세운",
        "연주",
        "year_pillar",
    ):
        if meta_row.get(key):
            stem, branch = pillar_chars(
                meta_row[key]
            )

            if stem and branch:
                return stem + branch

    return sexagenary_year_pillar(year)

def control_score(meta_row):
    candle = meta_row.get("candle") or {}
    value = candle.get("close")

    if value is None:
        return np.nan

    return float(value)

engine_by_id = {
    x["subject_id"]: x
    for x in new_engine["subjects"]
}

subject_by_id = {
    x["subject_id"]: x
    for x in new_corpus["subjects"]
}

calculator = sajupy.get_saju_calculator()
ENGINE_MIN_YEAR = int(calculator.min_year)
ENGINE_MAX_YEAR = int(calculator.max_year)

def build_subject_engine(subject):
    rec = engine_by_id[
        subject["subject_id"]
    ]

    birth = corrected_birth(
        rec["public_birth"]
    )

    if not (
        ENGINE_MIN_YEAR
        <= int(birth["y"])
        <= ENGINE_MAX_YEAR
    ):
        raise ValueError(
            "%s corrected birth year %s "
            "outside engine range %s-%s"
            % (
                subject["name"],
                birth["y"],
                ENGINE_MIN_YEAR,
                ENGINE_MAX_YEAR,
            )
        )

    inp = se.BirthInput(
        year=int(birth["y"]),
        month=int(birth["m"]),
        day=int(birth["d"]),
        hour=int(birth["h"]),
        minute=int(birth["min"]),
        gender=subject["gender"],
        calendar="solar",
        is_leap_month=False,
        use_solar_time=False,
        utc_offset=9,
    )

    result = se.compute_all(inp)

    timeline = result[
        "chart_data"
    ]["연도별_타임라인"]

    meta = {
        int(row["year"]): row
        for row in timeline
    }

    return result, meta

def extract_tengod(
    subject,
    year,
    result,
    meta_row,
):
    day_stem, _ = pillar_chars(
        result["원국"]["day"]
    )

    dw_stem, dw_branch = pillar_chars(
        get_daewoon_pillar(meta_row)
    )

    sw_stem, sw_branch = pillar_chars(
        get_sewoon_pillar(
            meta_row,
            year,
        )
    )

    if not all([
        day_stem,
        dw_stem,
        dw_branch,
        sw_stem,
        sw_branch,
    ]):
        raise RuntimeError(
            "Pillar parse failed: %s %s"
            % (subject["name"], year)
        )

    dw_group = ten_god_group(
        day_stem,
        dw_stem,
    )
    sw_group = ten_god_group(
        day_stem,
        sw_stem,
    )

    groups = [
        "peer",
        "output",
        "wealth",
        "officer",
        "resource",
    ]

    out = {}

    for group in groups:
        out[
            "tengod__dw_group_" + group
        ] = float(dw_group == group)

        out[
            "tengod__sw_group_" + group
        ] = float(sw_group == group)

    out["baseline__control_score"] = (
        control_score(meta_row)
    )

    return out

### Build/cache the 200 NEW_DEV_2 event-year feature rows

In [6]:
def build_feature_cache():
    if FEATURE_CACHE.exists():
        cached = pd.read_csv(
            FEATURE_CACHE
        )

        compatible = bool(
            cached["subject_id"].nunique() == 100
            and len(cached) == 200
            and set(TENGOD_COLS).issubset(
                cached.columns
            )
        )

        if compatible:
            print(
                "CACHE HIT:",
                FEATURE_CACHE,
            )
            return cached

        print(
            "Ignoring incompatible cache:",
            FEATURE_CACHE,
        )

    rows = []

    for i, (sid, group) in enumerate(
        new_targets.groupby("subject_id"),
        1,
    ):
        subject = subject_by_id[sid]
        result, meta = build_subject_engine(
            subject
        )

        for year in sorted(
            group["year"].unique()
        ):
            year = int(year)

            if year not in meta:
                raise RuntimeError(
                    "Annual timeline missing year "
                    "%s for %s"
                    % (year, subject["name"])
                )

            row = extract_tengod(
                subject,
                year,
                result,
                meta[year],
            )

            row["subject_id"] = sid
            row["year"] = year
            rows.append(row)

        if i % 10 == 0:
            print(
                "feature subjects %d/100"
                % i
            )

    out = pd.DataFrame(rows)

    missing = [
        c for c in TENGOD_COLS
        if c not in out.columns
    ]

    if missing:
        raise RuntimeError(
            "NEW_DEV_2 missing frozen "
            "features: %s" % missing
        )

    assert out["subject_id"].nunique() == 100
    assert len(out) == 200

    out.to_csv(
        FEATURE_CACHE,
        index=False,
    )

    print(
        "CACHED:",
        FEATURE_CACHE,
    )

    return out

new_features = build_feature_cache()

new_df = new_targets.merge(
    new_features,
    on=["subject_id", "year"],
    how="inner",
    validate="one_to_one",
)

assert new_df["subject_id"].nunique() == 100
assert len(new_df) == 200

print(
    "NEW_DEV_2 feature rows:",
    len(new_df),
)

[SAJU_DEBUG] original_input: 1941-05-24 19:56
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1941-05-24
[SAJU_DEBUG] final_datetime(KST): 1941-05-24T19:56:00+09:00
[SAJU_DEBUG] pillars: 연=辛巳 월=癸巳 일=壬申 시=庚戌
[SAJU_DEBUG] original_input: 1927-06-23 21:38
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1927-06-23
[SAJU_DEBUG] final_datetime(KST): 1927-06-23T21:38:00+09:00
[SAJU_DEBUG] pillars: 연=丁卯 월=丙午 일=戊子 시=癸亥
[SAJU_DEBUG] original_input: 1935-11-09 13:21
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1935-11-09
[SAJU_DEBUG] final_datetime(KST): 1935-11-09T13:21:00+09:00
[SAJU_DEBUG] 반시보정: 辛未→庚午 (13:21)
[SAJU_DEBUG] pillars: 연=乙亥 월=丁亥 일=己丑 시=庚午
[SAJU_DEBUG] original_input: 1942-12-20 14:02
[SAJU_DEBUG] calendar=solar, is_l

feature subjects 10/100


[SAJU_DEBUG] original_input: 1926-02-20 20:07
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1926-02-20
[SAJU_DEBUG] final_datetime(KST): 1926-02-20T20:07:00+09:00
[SAJU_DEBUG] pillars: 연=丙寅 월=庚寅 일=庚辰 시=丙戌
[SAJU_DEBUG] original_input: 1937-09-28 04:53
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1937-09-28
[SAJU_DEBUG] final_datetime(KST): 1937-09-28T04:53:00+09:00
[SAJU_DEBUG] pillars: 연=丁丑 월=己酉 일=戊午 시=甲寅
[SAJU_DEBUG] original_input: 1946-10-17 16:13
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1946-10-17
[SAJU_DEBUG] final_datetime(KST): 1946-10-17T16:13:00+09:00
[SAJU_DEBUG] pillars: 연=丙戌 월=戊戌 일=甲子 시=壬申
[SAJU_DEBUG] original_input: 1934-01-26 02:18
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

feature subjects 20/100


[SAJU_DEBUG] original_input: 1943-07-28 10:36
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1943-07-28
[SAJU_DEBUG] final_datetime(KST): 1943-07-28T10:36:00+09:00
[SAJU_DEBUG] pillars: 연=癸未 월=己未 일=丁亥 시=乙巳
[SAJU_DEBUG] original_input: 1922-06-22 06:39
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1922-06-22
[SAJU_DEBUG] final_datetime(KST): 1922-06-22T06:39:00+09:00
[SAJU_DEBUG] pillars: 연=壬戌 월=丙午 일=辛酉 시=辛卯
[SAJU_DEBUG] original_input: 1957-07-27 19:15
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1957-07-27
[SAJU_DEBUG] final_datetime(KST): 1957-07-27T19:15:00+09:00
[SAJU_DEBUG] 반시보정: 丙戌→乙酉 (19:15)
[SAJU_DEBUG] pillars: 연=丁酉 월=丁未 일=庚子 시=乙酉
[SAJU_DEBUG] original_input: 1946-07-29 09:42
[SAJU_DEBUG] calendar=solar, is_l

...

feature subjects 80/100


[SAJU_DEBUG] original_input: 1974-02-10 19:17
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1974-02-10
[SAJU_DEBUG] final_datetime(KST): 1974-02-10T19:17:00+09:00
[SAJU_DEBUG] 반시보정: 庚戌→己酉 (19:17)
[SAJU_DEBUG] pillars: 연=甲寅 월=丙寅 일=壬午 시=己酉
[SAJU_DEBUG] original_input: 1936-07-29 00:33
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1936-07-29
[SAJU_DEBUG] final_datetime(KST): 1936-07-29T00:33:00+09:00
[SAJU_DEBUG] pillars: 연=丙子 월=乙未 일=壬子 시=庚子
[SAJU_DEBUG] original_input: 1933-04-15 16:45
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1933-04-15
[SAJU_DEBUG] final_datetime(KST): 1933-04-15T16:45:00+09:00
[SAJU_DEBUG] pillars: 연=癸酉 월=丙辰 일=辛亥 시=丙申
[SAJU_DEBUG] original_input: 1989-02-16 18:56
[SAJU_DEBUG] calendar=solar

feature subjects 90/100


[SAJU_DEBUG] original_input: 1936-10-31 12:15
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1936-10-31
[SAJU_DEBUG] final_datetime(KST): 1936-10-31T12:15:00+09:00
[SAJU_DEBUG] pillars: 연=丙子 월=戊戌 일=丙戌 시=甲午
[SAJU_DEBUG] original_input: 1980-05-18 08:18
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1980-05-18
[SAJU_DEBUG] final_datetime(KST): 1980-05-18T08:18:00+09:00
[SAJU_DEBUG] pillars: 연=庚申 월=辛巳 일=辛卯 시=壬辰
[SAJU_DEBUG] original_input: 1955-10-20 08:02
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1955-10-20
[SAJU_DEBUG] final_datetime(KST): 1955-10-20T08:02:00+09:00
[SAJU_DEBUG] pillars: 연=乙未 월=丙戌 일=甲寅 시=戊辰
[SAJU_DEBUG] original_input: 1962-08-13 18:36
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

feature subjects 100/100
CACHED: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v3_new_dev_2_replication/NEW_DEV_2_feature_table_v3.csv
NEW_DEV_2 feature rows: 200


## 5. One-shot frozen scoring

In [7]:
scaler = MODEL.named_steps["scale"]
classifier = MODEL.named_steps["model"]

X = new_df[
    TENGOD_COLS
].to_numpy(dtype=float)

new_df["tengod_score"] = (
    scaler.transform(X)
    .dot(
        classifier.coef_.ravel()
    )
)

subject_rows = []

for sid, group in new_df.groupby(
    "subject_id"
):
    positive = group[
        group["label"] == 1
    ].iloc[0]

    negative = group[
        group["label"] == 0
    ].iloc[0]

    if (
        positive["tengod_score"]
        > negative["tengod_score"]
    ):
        model_pairwise = 1.0
    elif (
        positive["tengod_score"]
        == negative["tengod_score"]
    ):
        model_pairwise = 0.5
    else:
        model_pairwise = 0.0

    pc = positive[
        "baseline__control_score"
    ]
    nc = negative[
        "baseline__control_score"
    ]

    if pd.isna(pc) or pd.isna(nc):
        control_pairwise = np.nan
    elif pc > nc:
        control_pairwise = 1.0
    elif pc == nc:
        control_pairwise = 0.5
    else:
        control_pairwise = 0.0

    subject_rows.append({
        "subject_id": sid,
        "name": positive["name"],
        "gender": positive["gender"],
        "occupation_category": positive[
            "occupation_category"
        ],
        "positive_year": int(
            positive["year"]
        ),
        "negative_year": int(
            negative["year"]
        ),
        "positive_confidence": positive[
            "confidence"
        ],
        "negative_confidence": negative[
            "confidence"
        ],
        "tengod_pairwise": model_pairwise,
        "control_pairwise": control_pairwise,
        "score_separation": float(
            positive["tengod_score"]
            - negative["tengod_score"]
        ),
    })

subject_df = pd.DataFrame(
    subject_rows
)

macro_pairwise = float(
    subject_df[
        "tengod_pairwise"
    ].mean()
)

strict_hit_rate = float(
    (
        subject_df["tengod_pairwise"]
        > 0.5
    ).mean()
)

control_valid = subject_df[
    subject_df[
        "control_pairwise"
    ].notna()
].copy()

control_pairwise = float(
    control_valid[
        "control_pairwise"
    ].mean()
)

control_valid[
    "delta_vs_control"
] = (
    control_valid[
        "tengod_pairwise"
    ]
    - control_valid[
        "control_pairwise"
    ]
)

delta_vs_control = float(
    control_valid[
        "delta_vs_control"
    ].mean()
)

print(
    "TENGOD macro pairwise:",
    macro_pairwise,
)
print(
    "TENGOD strict hit:",
    strict_hit_rate,
)
print(
    "Control pairwise:",
    control_pairwise,
)
print(
    "Delta vs Control:",
    delta_vs_control,
)
print(
    "Control-evaluable subjects:",
    len(control_valid),
)

new_df.to_csv(
    OUT / "NEW_DEV_2_scored_rows_v3.csv",
    index=False,
)

subject_df.to_csv(
    OUT / "NEW_DEV_2_subject_results_v3.csv",
    index=False,
)

TENGOD macro pairwise: 0.56
TENGOD strict hit: 0.53
Control pairwise: 0.55
Delta vs Control: 0.01
Control-evaluable subjects: 100


## 6. Fixed-model label permutation

The frozen model is never refit during permutation.

With one positive and one negative year per person, the null randomly swaps
the two labels independently for each subject.

In [8]:
rng = np.random.RandomState(
    SEED + 100
)

observed = macro_pairwise
null = []

score_pairs = []

for sid, group in new_df.groupby(
    "subject_id"
):
    positive = group[
        group["label"] == 1
    ]["tengod_score"].iloc[0]

    negative = group[
        group["label"] == 0
    ]["tengod_score"].iloc[0]

    score_pairs.append(
        (float(positive), float(negative))
    )

for _ in range(N_PERMUTATIONS):
    values = []

    for original_positive, original_negative in score_pairs:
        if rng.rand() < 0.5:
            p_score = original_positive
            n_score = original_negative
        else:
            p_score = original_negative
            n_score = original_positive

        if p_score > n_score:
            values.append(1.0)
        elif p_score == n_score:
            values.append(0.5)
        else:
            values.append(0.0)

    null.append(
        float(np.mean(values))
    )

null = np.asarray(null)

permutation_p = (
    1.0
    + float(
        np.sum(null >= observed)
    )
) / (
    len(null) + 1.0
)

permutation_df = pd.DataFrame([{
    "observed_pairwise": observed,
    "null_mean": float(
        null.mean()
    ),
    "null_p95": float(
        np.quantile(
            null,
            0.95,
        )
    ),
    "n_permutations": N_PERMUTATIONS,
    "p_value": permutation_p,
}])

display(permutation_df)

permutation_df.to_csv(
    OUT / "NEW_DEV_2_permutation_v3.csv",
    index=False,
)

,observed_pairwise,null_mean,null_p95,n_permutations,p_value
0,0.56,0.500332,0.58,5000,0.119176


## 7. Subject bootstrap vs chance and Production Control

In [9]:
rng = np.random.RandomState(
    SEED + 200
)

pairwise_values = subject_df[
    "tengod_pairwise"
].to_numpy(dtype=float)

pair_boot = np.asarray([
    rng.choice(
        pairwise_values,
        size=len(pairwise_values),
        replace=True,
    ).mean()
    for _ in range(N_BOOTSTRAP)
])

delta_values = control_valid[
    "delta_vs_control"
].to_numpy(dtype=float)

delta_boot = np.asarray([
    rng.choice(
        delta_values,
        size=len(delta_values),
        replace=True,
    ).mean()
    for _ in range(N_BOOTSTRAP)
])

bootstrap_p_pairwise_gt_half = float(
    np.mean(
        pair_boot > 0.5
    )
)

bootstrap_p_delta_gt_zero = float(
    np.mean(
        delta_boot > 0.0
    )
)

bootstrap_df = pd.DataFrame([{
    "macro_pairwise": macro_pairwise,
    "pairwise_ci_low": float(
        np.quantile(
            pair_boot,
            0.025,
        )
    ),
    "pairwise_ci_high": float(
        np.quantile(
            pair_boot,
            0.975,
        )
    ),
    "bootstrap_p_pairwise_gt_0_5": (
        bootstrap_p_pairwise_gt_half
    ),
    "control_pairwise": (
        control_pairwise
    ),
    "n_control_evaluable": int(
        len(control_valid)
    ),
    "delta_vs_control": (
        delta_vs_control
    ),
    "delta_ci_low": float(
        np.quantile(
            delta_boot,
            0.025,
        )
    ),
    "delta_ci_high": float(
        np.quantile(
            delta_boot,
            0.975,
        )
    ),
    "bootstrap_p_delta_gt_0": (
        bootstrap_p_delta_gt_zero
    ),
}])

display(bootstrap_df)

bootstrap_df.to_csv(
    OUT / "NEW_DEV_2_bootstrap_v3.csv",
    index=False,
)

,macro_pairwise,pairwise_ci_low,pairwise_ci_high,bootstrap_p_pairwise_gt_0_5,control_pairwise,n_control_evaluable,delta_vs_control,delta_ci_low,delta_ci_high,bootstrap_p_delta_gt_0
0,0.56,0.465,0.655,0.88495,0.55,100,0.01,-0.125,0.14,0.54545


## 8. Sensitivity slices

In [10]:
high_ids = []

for sid, group in new_df.groupby(
    "subject_id"
):
    if (
        len(group) == 2
        and all(
            x == "high"
            for x in group[
                "confidence"
            ].tolist()
        )
    ):
        high_ids.append(sid)

sensitivity_rows = []

high_subjects = subject_df[
    subject_df[
        "subject_id"
    ].isin(high_ids)
]

if len(high_subjects):
    sensitivity_rows.append({
        "slice": (
            "BOTH_EVENTS_HIGH_CONFIDENCE"
        ),
        "n_subjects": int(
            len(high_subjects)
        ),
        "tengod_pairwise": float(
            high_subjects[
                "tengod_pairwise"
            ].mean()
        ),
        "control_pairwise": float(
            high_subjects[
                "control_pairwise"
            ].mean()
        ),
    })

for occupation, group in subject_df.groupby(
    "occupation_category"
):
    sensitivity_rows.append({
        "slice": (
            "OCCUPATION_"
            + str(occupation)
        ),
        "n_subjects": int(
            len(group)
        ),
        "tengod_pairwise": float(
            group[
                "tengod_pairwise"
            ].mean()
        ),
        "control_pairwise": float(
            group[
                "control_pairwise"
            ].mean()
        ),
    })

for gender, group in subject_df.groupby(
    "gender"
):
    sensitivity_rows.append({
        "slice": (
            "GENDER_"
            + str(gender)
        ),
        "n_subjects": int(
            len(group)
        ),
        "tengod_pairwise": float(
            group[
                "tengod_pairwise"
            ].mean()
        ),
        "control_pairwise": float(
            group[
                "control_pairwise"
            ].mean()
        ),
    })

sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

display(sensitivity_df)

sensitivity_df.to_csv(
    OUT / "NEW_DEV_2_sensitivity_v3.csv",
    index=False,
)

,slice,n_subjects,tengod_pairwise,control_pairwise
0,BOTH_EVENTS_HIGH_CONFIDENCE,65,0.553846,0.576923
1,OCCUPATION_arts_entertainment,37,0.581081,0.567568
2,OCCUPATION_fashion,1,1.000000,1.000000
3,OCCUPATION_intelligence,1,0.000000,1.000000
4,OCCUPATION_journalism,1,1.000000,1.000000
5,OCCUPATION_politics,8,0.812500,0.875000
6,OCCUPATION_sports,51,0.490196,0.470588
7,OCCUPATION_sports_politics,1,1.000000,0.000000
8,GENDER_female,14,0.642857,0.500000
9,GENDER_male,86,0.546512,0.558140


# 9. Predeclared replication decision

## Primary replication pass

All must hold:

1. `macro_pairwise >= 0.55`
2. fixed-model permutation `p < 0.05`
3. bootstrap `P(pairwise > 0.5) >= 0.95`

This answers whether the frozen Ten-God signal replicates above chance.

## Separate Production Control support

Also report:

1. mean delta vs Control > 0
2. bootstrap `P(delta > 0) >= 0.90`

Control superiority is **not** substituted into the primary replication gate.

## After the result

A failure does not authorize tuning on NEW_DEV_2.
A pass does not automatically open NEW_CONFIRM: first review source/data QA.

In [11]:
replication_pass = bool(
    macro_pairwise >= 0.55
    and permutation_p < 0.05
    and bootstrap_p_pairwise_gt_half >= 0.95
)

control_superiority_support = bool(
    delta_vs_control > 0
    and bootstrap_p_delta_gt_zero >= 0.90
)

if (
    replication_pass
    and control_superiority_support
):
    status = (
        "REPLICATION_PASS_WITH_"
        "CONTROL_SUPERIORITY_SUPPORT"
    )
elif replication_pass:
    status = (
        "REPLICATION_PASS_SIGNAL_ONLY"
    )
else:
    status = (
        "REPLICATION_FAIL_"
        "DO_NOT_TUNE_ON_NEW_DEV_2"
    )

summary = pd.DataFrame([{
    "status": status,
    "n_subjects": 100,
    "macro_pairwise": (
        macro_pairwise
    ),
    "strict_hit_rate": (
        strict_hit_rate
    ),
    "permutation_p": (
        permutation_p
    ),
    "bootstrap_p_pairwise_gt_0_5": (
        bootstrap_p_pairwise_gt_half
    ),
    "control_pairwise": (
        control_pairwise
    ),
    "n_control_evaluable": int(
        len(control_valid)
    ),
    "delta_vs_control": (
        delta_vs_control
    ),
    "bootstrap_p_delta_gt_0": (
        bootstrap_p_delta_gt_zero
    ),
    "replication_pass": (
        replication_pass
    ),
    "control_superiority_support": (
        control_superiority_support
    ),
}])

display(summary)

summary.to_csv(
    OUT / "NEW_DEV_2_replication_summary_v3.csv",
    index=False,
)

decision = {
    "version": (
        "NEW_DEV_2_REPLICATION_"
        "DECISION_V3"
    ),
    "notebook_version": (
        NOTEBOOK_VERSION
    ),
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
    "status": status,
    "frozen_architecture": {
        "winner": freeze[
            "winner"
        ],
        "features": TENGOD_COLS,
        "model": freeze[
            "winner_parameters"
        ],
    },
    "n_subjects": 100,
    "metrics": {
        key: (
            value.item()
            if hasattr(value, "item")
            else value
        )
        for key, value in summary.iloc[
            0
        ].to_dict().items()
    },
    "integrity": {
        "NEW_DEV_2_corpus_sha256": (
            sha256_file(
                NEW_CORPUS_PATH
            )
        ),
        "NEW_DEV_2_engine_inputs_sha256": (
            sha256_file(
                NEW_ENGINE_PATH
            )
        ),
        "architecture_freeze_sha256": (
            sha256_file(
                FREEZE_PATH
            )
        ),
        "NEW_CONFIRM_loaded": False,
    },
    "next_rule": (
        "If replication passes, review event-source QA before deciding "
        "whether to open sealed V2 NEW_CONFIRM one-shot. "
        "If replication fails, do not tune on NEW_DEV_2."
    ),
}

with open(
    OUT / "NEW_DEV_2_REPLICATION_DECISION_V3.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        decision,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    json.dumps(
        decision,
        ensure_ascii=False,
        indent=2,
    )
)

,status,n_subjects,macro_pairwise,strict_hit_rate,permutation_p,bootstrap_p_pairwise_gt_0_5,control_pairwise,n_control_evaluable,delta_vs_control,bootstrap_p_delta_gt_0,replication_pass,control_superiority_support
0,REPLICATION_FAIL_DO_NOT_TUNE_ON_NEW_DEV_2,100,0.56,0.53,0.119176,0.88495,0.55,100,0.01,0.54545,False,False


{
  "version": "NEW_DEV_2_REPLICATION_DECISION_V3",
  "notebook_version": "SAJU_ML_NEW_DEV_2_FROZEN_TENGOD_REPLICATION_V3_20260815",
  "created_at": "2026-08-15T22:00:38",
  "status": "REPLICATION_FAIL_DO_NOT_TUNE_ON_NEW_DEV_2",
  "frozen_architecture": {
    "winner": "TENGOD_FIXED",
    "features": [
      "tengod__dw_group_peer",
      "tengod__dw_group_output",
      "tengod__dw_group_wealth",
      "tengod__dw_group_officer",
      "tengod__dw_group_resource",
      "tengod__sw_group_peer",
      "tengod__sw_group_output",
      "tengod__sw_group_wealth",
      "tengod__sw_group_officer",
      "tengod__sw_group_resource"
    ],
    "model": {
      "model": "Pairwise Logistic Regression",
      "penalty": "l2",
      "C": 0.3
    }
  },
  "n_subjects": 100,
  "metrics": {
    "status": "REPLICATION_FAIL_DO_NOT_TUNE_ON_NEW_DEV_2",
    "n_subjects": 100,
    "macro_pairwise": 0.56,
    "strict_hit_rate": 0.53,
    "permutation_p": 0.1191761647670466,
    "bootstrap_p_pairwise_gt_0_

## Send these outputs back

After **Restart Kernel -> Run All**:

```text
research/ml/artifacts/v3_new_dev_2_replication/
├── NEW_DEV_2_replication_summary_v3.csv
├── NEW_DEV_2_permutation_v3.csv
├── NEW_DEV_2_bootstrap_v3.csv
├── NEW_DEV_2_subject_results_v3.csv
├── NEW_DEV_2_sensitivity_v3.csv
└── NEW_DEV_2_REPLICATION_DECISION_V3.json
```

Do not open `V2_NEW_CONFIRM` before we review this one-shot result.